In [ ]:
# PyTorch 라이브러리 임포트
import torch
# 신경망 모듈 임포트
import torch.nn as nn
# 최적화 알고리즘 임포트
import torch.optim as optim
# 데이터 로더와 텐서 데이터셋 임포트
from torch.utils.data import DataLoader, TensorDataset
# 데이터셋과 변환 도구 임포트
from torchvision import datasets, transforms

# Apple Silicon GPU 사용 가능 여부 확인
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print(f"Using Apple device : device={device}")
else:
    device = torch.device("cpu")
    print(f"Using CPU device : device={device}")    

# 데이터 전처리 파이프라인 정의
transform = transforms.Compose([
    # 이미지를 PyTorch 텐서로 변환
    transforms.ToTensor(),
    # MNIST 데이터셋의 정규화 (평균과 표준편차 적용)
    transforms.Normalize((0.1307,), (0.3081,))
])

# 학습용 MNIST 데이터셋 다운로드 및 로드
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
# 테스트용 MNIST 데이터셋 다운로드 및 로드
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)   

# 배치 크기 설정
batch_size = 64
# 학습 데이터 로더 생성 (섞기 활성화)
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
# 테스트 데이터 로더 생성 (섞기 비활성화)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)    

# 간단한 분류기 모델 클래스 정의
class SimpleClassifier(nn.Module):
    def __init__(self):
        # 부모 클래스 초기화
        super(SimpleClassifier, self).__init__()
        # 입력 이미지를 1차원으로 평탄화
        self.flatten = nn.Flatten()
        # 신경망 레이어 구성
        self.layer_stack = nn.Sequential(
            # 첫 번째 선형 레이어: 784(28*28) -> 512
            nn.Linear(28*28, 512),
            # 활성화 함수 ReLU
            nn.ReLU(),
            # 두 번째 선형 레이어: 512 -> 512
            nn.Linear(512, 512),
            # 활성화 함수 ReLU
            nn.ReLU(),
            # 출력 레이어: 512 -> 10 (숫자 클래스)
            nn.Linear(512, 10)
        )

    # 순전파 함수 정의
    def forward(self, x):
        # 입력 데이터 평탄화
        x = self.flatten(x)
        # 레이어 스택을 통과시켜 로짓 반환
        logits = self.layer_stack(x)
        return logits

# 모델 인스턴스 생성 및 디바이스에 할당
model = SimpleClassifier().to(device)
# 손실 함수 정의 (크로스 엔트로피)
criterion = nn.CrossEntropyLoss()
# Adam 옵티마이저 설정 (학습률 0.001)
optimizer = optim.Adam(model.parameters(), lr=0.001)
# 총 학습 에포크 수 설정
num_epochs = 5

# 모델 학습 시작
for epoch in range(num_epochs):
    # 모델을 학습 모드로 설정
    model.train()
    # 에포크 손실값 초기화
    running_loss = 0.0
    # 배치 단위로 데이터 처리
    for images, labels in train_loader:
        # 데이터를 지정된 디바이스로 이동
        images, labels = images.to(device), labels.to(device)

        # 모델을 통한 예측 수행
        outputs = model(images)
        # 손실값 계산
        loss = criterion(outputs, labels)

        # 기존 그래디언트 초기화
        optimizer.zero_grad()
        # 역전파 수행
        loss.backward()
        # 가중치 업데이트
        optimizer.step()

        # 배치의 손실값 누적
        running_loss += loss.item() * images.size(0)

    # 에포크의 평균 손실값 계산 및 출력
    epoch_loss = running_loss / len(train_loader.dataset)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}')    

# 모델 평가 시작
model.eval()
# 그래디언트 계산 비활성화
with torch.no_grad():
    # 정확도 계산을 위한 변수 초기화
    correct = 0
    total = 0
    # 테스트 데이터 처리
    for images, labels in test_loader:
        # 데이터를 지정된 디바이스로 이동
        images, labels = images.to(device), labels.to(device)
        # 예측 수행
        outputs = model(images)
        # 가장 높은 확률의 클래스 선택
        _, predicted = torch.max(outputs.data, 1)
        # 전체 샘플 수 누적
        total += labels.size(0)
        # 정확히 예측한 샘플 수 누적
        correct += (predicted == labels).sum().item()

    # 최종 정확도 계산 및 출력
    print(f'Accuracy of the model on the test images: {100 * correct / total:.2f}%')    

# 학습된 모델의 가중치를 파일로 저장
torch.save(model.state_dict(), 'simple_classifier.pth')

Using Apple device : device=mps
Epoch [1/5], Loss: 0.1982
Epoch [2/5], Loss: 0.0863
Epoch [3/5], Loss: 0.0629
Epoch [4/5], Loss: 0.0476
Epoch [5/5], Loss: 0.0380
Accuracy of the model on the test images: 97.37%
